# Campaign outcomes per trained model version

Every campaign in `D:/twdata/runs/human` is assigned to the model version live when it
started: the latest retrain (from the `session_*.json` reports) before its first decision
timestamp. `trained_at`/`rows` identify the version; both are null for campaigns played
before any model was fitted.

In [ ]:
import glob, json, os, sqlite3, time
import pandas as pd

RUNS = r"D:\twdata\runs\human"

retrains = []
for rp in sorted(glob.glob(os.path.join(RUNS, "session_*.json"))):
    rep = json.load(open(rp, encoding="utf-8"))
    for c in rep.get("campaigns") or []:
        r = c.get("retrain")
        if r and r.get("trained"):
            retrains.append({"ts": c["started"], "rows": r.get("rows")})
retrains.sort(key=lambda r: r["ts"])

def version_of(ts):
    ver = (float("nan"), float("nan"))
    for r in retrains:
        if ts >= r["ts"]:
            ver = (r["ts"], r["rows"])
    return ver

camps = {}
for db in sorted(glob.glob(os.path.join(RUNS, "*", "decisions.sqlite"))):
    con = sqlite3.connect("file:%s?mode=ro" % db.replace("\\", "/"), uri=True)
    try:
        for cid, t0 in con.execute(
                "SELECT campaign_id, MIN(ts) FROM decision_points GROUP BY campaign_id"):
            camps.setdefault(cid, {}).update(start_ts=t0)
        for cid, n, setts, lvl in con.execute(
                "SELECT campaign_id, COUNT(*), MAX(settlements), MAX(lord_level)"
                " FROM target_rows GROUP BY campaign_id"):
            camps.setdefault(cid, {}).update(turns_played=n, max_settlements=setts,
                                             max_lord_level=lvl)
    finally:
        con.close()

per_camp = pd.DataFrame([dict(campaign=cid, trained_ts=version_of(c["start_ts"])[0],
                              rows=version_of(c["start_ts"])[1], **c)
                         for cid, c in camps.items() if c.get("start_ts")])
df = (per_camp.groupby(["trained_ts", "rows"], dropna=False)
      .agg(campaigns=("campaign", "count"),
           avg_turns_played=("turns_played", "mean"),
           avg_max_settlements=("max_settlements", "mean"),
           best_settlements=("max_settlements", "max"),
           avg_max_lord_level=("max_lord_level", "mean"),
           best_lord_level=("max_lord_level", "max"))
      .round(2)
      .reset_index()
      .sort_values("trained_ts", na_position="first")
      .reset_index(drop=True))
df.insert(0, "trained_at", df.pop("trained_ts").map(
    lambda t: time.strftime("%Y-%m-%d %H:%M", time.localtime(t)) if t == t else None))

In [2]:
df['settlement_expansion_rate']=df['avg_max_settlements']/df['avg_turns_played']
df['Legendary_lord_level_rate']=df['avg_max_lord_level']/df['avg_turns_played']
df

,campaigns,avg_turns_played,avg_max_settlements,best_settlements,avg_max_lord_level,best_lord_level,settlement_expansion_rate,Legendary_lord_level_rate
model_version,,,,,,,,
cold (no model),10,5.60,1.20,2.0,2.00,4.0,0.214286,0.357143
trained 08-03 18:00 (438 rows),4,9.75,1.25,2.0,1.75,2.0,0.128205,0.179487
trained 08-03 19:23 (778 rows),3,4.00,1.67,2.0,2.00,3.0,0.417500,0.500000
